# Notebook 10 – Feature Selection
## What is Feature Selection?

Feature selection is choosing which columns actually help a model make
better predictions, and removing the ones that don't. This is different
from feature engineering (creating new columns) and encoding (converting
categories to numbers), it happens after those steps, right before
modeling.

## Why Feature Selection?

More columns doesn't mean a better model. It often means the opposite.
Unnecessary features:

* Add noise the model has to learn to ignore, which it doesn't always do well.
* Slow down training, sometimes dramatically with high-cardinality or
  one-hot encoded columns.
* Increase the risk of **overfitting**, the model starts memorizing quirks
  of columns that have nothing to do with the real pattern, and performs
  worse on new, unseen data.
* Make the model harder to interpret, if a stakeholder asks "why did the
  model predict this," a bloated feature list makes that answer messier.



In [8]:
import pandas as pd
import glob
import os

# Automatically locate the CSV, even if it's not in the current working directory
matches = glob.glob("**/hr_employee_attrition_raw.csv", recursive=True)

if matches:
    file_path = matches[0]
    print("Found file at:", file_path)
else:
    raise FileNotFoundError(
        "Could not find 'hr_employee_attrition_raw.csv' anywhere under: "
        + os.getcwd()
        + "\nMake sure the file is saved somewhere inside this folder or a subfolder."
    )

df = pd.read_csv(file_path)

df["MonthlyIncome_clean"] = pd.to_numeric(
    df["MonthlyIncome"].astype(str).str.replace(" USD", "", regex=False).str.strip(),
    errors="coerce"
)
df["Attrition_binary"] = df["Attrition"].map({"Yes": 1, "No": 0})
df.shape

Found file at: Downloads\hr_employee_attrition_raw.csv


(1230, 18)

## Relevant, Irrelevant, and Redundant Features

* **Relevant:** actually helps predict the target. `MonthlyIncome`,
  `JobSatisfaction`, and `YearsAtCompany` are reasonable candidates for
  predicting `Attrition`.
* **Irrelevant:** has no real relationship with the target, but isn't
  necessarily useless for other purposes. `RandomSurveyCode` in our
  dataset looks like an internal tracking number, unrelated to attrition.
* **Redundant:** carries the same information as another column already in
  the dataset. If we had both `YearsAtCompany` and `JoinDate`, they'd be
  redundant since one can be derived from the other.



## Variance Threshold

**What it does:** removes features that barely change across rows. A
column with almost no variance can't possibly help distinguish one
employee from another.

**Real example from this data:** `EmployeeCountFlag` has exactly **one
unique value** across all 1,230 rows, its variance is literally **0**.
It's a constant, it can never help a model split or separate employees,
no matter how it's encoded or scaled.

In [9]:
from sklearn.feature_selection import VarianceThreshold

print("EmployeeCountFlag unique values:", df["EmployeeCountFlag"].nunique())
print("EmployeeCountFlag variance:", df["EmployeeCountFlag"].var())

# VarianceThreshold will flag and remove zero-variance columns automatically
numeric_cols = df[["Age", "MonthlyIncome_clean", "EmployeeCountFlag"]].dropna()
selector = VarianceThreshold(threshold=0.0)
selector.fit(numeric_cols)
print("Columns kept:", numeric_cols.columns[selector.get_support()].tolist())

EmployeeCountFlag unique values: 1
EmployeeCountFlag variance: 0.0
Columns kept: ['Age', 'MonthlyIncome_clean']


## Correlation-Based Selection

**What it does:** checks how strongly each numeric feature relates to the
target (or to other features), keeping the strong ones and flagging the
weak or redundant ones.

**Real example from this data:** checking correlation with `Attrition`
shows every numeric column we have sits close to **zero correlation**, none
of them individually predict attrition well on their own using a simple
linear relationship. This doesn't necessarily mean they're all useless
(some patterns are non-linear and won't show up in a correlation
coefficient), but it's a useful first filter.

In [10]:
numeric_cols = ["Age", "MonthlyIncome_clean", "JobSatisfaction", "PerformanceRating",
                 "DistanceFromHomeKM", "RandomSurveyCode", "Attrition_binary"]

correlations = df[numeric_cols].dropna().corr()["Attrition_binary"].sort_values()
correlations

RandomSurveyCode      -0.017726
PerformanceRating     -0.011444
MonthlyIncome_clean    0.000667
DistanceFromHomeKM     0.000700
JobSatisfaction        0.012573
Age                    0.038944
Attrition_binary       1.000000
Name: Attrition_binary, dtype: float64

## Univariate Feature Selection

**What it does:** scores each feature individually against the target
using a statistical test (like chi-squared or ANOVA F-test), then keeps
only the top-scoring features. It's called "univariate" because it looks
at one feature at a time, ignoring interactions between features.

**Risk of relying on it alone:** it can miss features that only matter
**in combination** with another feature. A feature might score poorly on
its own but become very informative when paired with a second one.

In [11]:
from sklearn.feature_selection import SelectKBest, f_classif

model_df = df[["Age", "MonthlyIncome_clean", "JobSatisfaction", "PerformanceRating",
                "DistanceFromHomeKM", "Attrition_binary"]].dropna()

X = model_df.drop(columns=["Attrition_binary"])
y = model_df["Attrition_binary"]

selector = SelectKBest(score_func=f_classif, k=3)
selector.fit(X, y)

scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
scores

Age                    1.664742
JobSatisfaction        0.173288
PerformanceRating      0.143547
DistanceFromHomeKM     0.000537
MonthlyIncome_clean    0.000488
dtype: float64

## Mutual Information

**What it does:** measures how much knowing one feature reduces
uncertainty about the target, without assuming a linear relationship like
correlation does. It can catch patterns correlation would completely miss.

**Why it matters here:** since our correlation check showed everything
near zero, mutual information is a useful second opinion, it might reveal
a non-linear relationship that a simple correlation coefficient can't see.

In [12]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(X, y, random_state=42)
pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

MonthlyIncome_clean    0.018847
DistanceFromHomeKM     0.000219
Age                    0.000000
JobSatisfaction        0.000000
PerformanceRating      0.000000
dtype: float64

## Recursive Feature Elimination (RFE)

**What it does:** trains a model, ranks features by importance, removes
the weakest one, then repeats, recursively narrowing down to the best
subset. Unlike univariate selection, this accounts for how features
interact with each other during training.

**Trade-off:** more accurate than univariate selection, but much more
computationally expensive, since it retrains the model repeatedly.

In [13]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
rfe = RFE(model, n_features_to_select=3)
rfe.fit(X, y)

selected_features = X.columns[rfe.support_].tolist()
print("Features selected by RFE:", selected_features)

Features selected by RFE: ['Age', 'JobSatisfaction', 'PerformanceRating']


## Feature Importance (Model-Based)

**What it does:** trains a tree-based model (like Random Forest) and reads
off how much each feature contributed to reducing prediction error across
all the trees. This naturally captures interactions and non-linear effects.

**Real example from this data:** if `RandomSurveyCode` shows up with
meaningful importance here, that's a red flag, not a discovery. It suggests
the model is overfitting to noise, since that column is described as
random and shouldn't logically predict attrition at all.

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42, n_estimators=200)
rf.fit(X, y)

importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importance

MonthlyIncome_clean    0.338747
DistanceFromHomeKM     0.285495
Age                    0.221987
JobSatisfaction        0.078215
PerformanceRating      0.075556
dtype: float64

## How Unnecessary Features Actually Hurt a Model

Bringing it all together, here's what including columns like
`EmployeeCountFlag` and `RandomSurveyCode` would actually do to a model:

* **Overfitting:** a model might find a coincidental pattern in
  `RandomSurveyCode` (since it's a unique number per employee) and use it
  to "predict" attrition on the training data, purely by memorization. That
  pattern won't exist in new data, so accuracy on unseen employees drops.
* **Wasted capacity:** `EmployeeCountFlag` contributes zero information but
  still takes up a column, memory, and processing time in every single
  calculation.
* **Diluted signal:** if a model has to weigh 20 features and only 5 are
  meaningful, algorithms like KNN or linear models can get pulled off
  track by the noisy 15, especially before scaling and selection are done.
* **Harder debugging:** when a model underperforms, having irrelevant
  features in the mix makes it harder to isolate whether the problem is
  the data, the model, or a specific noisy column.